In [ ]:
import pandas as pd
from collections import defaultdict
import numpy as np
import sys


proteomicsDataColumns = ["Majority protein IDs", 
                        "Gene names", "Q-value", 
                        "Score", 
                        "Intensity \d+_\d+", 
                        "LFQ intensity \d+_\d+", 
                        "Glycation site positions", 
                        "Glycation site positions",
                        "Reverse",
                        "Potential contaminant",
                        "Only identified by site"]

proteomicsDataFilters = ["Reverse",
                       "Potential contaminant",
                       "Only identified by site"]

proteomicsDataProteinCol = "Majority protein IDs"

proteomicsDataGeneCol = "Gene names"
proteomicsDataModPosCol = ["Glycation site positions", "Oxidation (M) site positions"]
proteomicsDataLog = "log10"

def loadProteomicsDataset(uri):
    ''' This function gets the molecular data from a proteomics experiment.
        Input: uri of the processed file resulting from MQ
        Output: pandas DataFrame with the columns and filters defined in config.py '''

    aux = None
    
    #Get the columns from config and divide them into simple or regex columns
    columns = proteomicsDataColumns
    regexCols = [c for c in columns if '+' in c]
    columns = set(columns).difference(regexCols)
        
    #Read the filters defined in config, i.e. reverse, contaminant, etc.
    filters = proteomicsDataFilters
    proteinCol = proteomicsDataProteinCol
    geneCol = proteomicsDataGeneCol
    modPosCol = proteomicsDataModPosCol
    
    
    #Read the data from file
    data = pd.read_csv(uri, sep = '\t', low_memory=False)

    #Apply filters
    data = data[data[filters].isnull().any(1)]
    data = data.drop(filters, axis=1)
    
    #Select first protein form group, i.e. P01911;Q29830;Q9MXZ4 -> P01911
    #Set protein as index
    proteins = data[proteinCol].str.split(';').apply(pd.Series,1)[0]
    data[proteinCol] = proteins
    data = data.set_index(proteinCol)
    filters.append(proteinCol)
    columns = set(columns).difference(filters)
    
    #As well, select first gene name and first modification position corresponding
    #to the selected protein
    genes = data[geneCol].str.split(';').apply(pd.Series,1)[0]
    data[geneCol] = genes
    for modCol in modPosCol:
        mod = data[modCol].str.split(';').apply(pd.Series,1)[0]
        data[modCol] = mod

    #Get columns using regex
    for regex in regexCols:
        if aux is None:
            aux = data.filter(regex=regex)
        else:
            aux = aux.join(data.filter(regex=regex))

    #Add simple and regex columns into a single DataFrame
    data = data[list(columns)]
    data = data.join(aux)

    return data, regexCols

def extractSubjectReplicates(data, regex):
    subjectDict = defaultdict(list)
    for r in regex:
        columns = data.filter(regex = r).columns
        for c in columns:
            fields  = c.split('_')
            value = " ".join(fields[0].split(' ')[0:-1])
            subject = fields[1]
            ident = value + " " + subject 
            subjectDict[ident].append(c)

    return subjectDict
            
def parseProteomicsDataset(projectID, uri):
    log = proteomicsDataLog
    data, regex = loadProteomicsDataset(uri)
    subjectDict = extractSubjectReplicates(data, regex)
    delCols = []
    for subject in subjectDict:
            delCols.extend(subjectDict[subject])
            aux = data[subjectDict[subject]]
            data[subject] = calculateMedianReplicates(aux, log)

    data = data.drop(delCols, 1)
    
    
    return data

def calculateMedianReplicates(data, log = "log2"):
    if log == "log2":
        data = data.applymap(lambda x:np.log2(x) if x > 0 else np.nan)
    elif log == "log10":
        data = data.applymap(lambda x:np.log10(x) if x > 0 else np.nan)
    median = data.median(axis=1).sort_values(axis=0, ascending= True, na_position = 'first').to_frame()
    return median


def extractProteinSubjectRelationships(data):
    aux =  data.filter(regex = 'LFQ intensity')
    aux.columns = [c.split(" ")[2] for c in aux.columns]
    aux = aux.stack()
    aux = aux.reset_index()
    aux.columns = [':START_ID(Protein)', ':END_ID(Subject)', "LFQ intensity"]
    
    return aux

#ToDo
def extractModificationProteinSubjectRelationships(data):
    subjects =  data.filter(regex = 'LFQ intensity')
    subjects.columns = [c.split(" ")[2] for c in subjects.columns]
    subjects = subjects.stack()
    subjects = subjects.reset_index()
    icol = subjects.columns[0]
    subjects.columns  = [icol, "subject id","LFQ intensity"]
    
    modPosCol = proteomicsDataModPosCol
    cols = [c for c in data.columns if c in modPosCol]
    mod =  data.filter(cols)
    mod.columns = [modPosCol[c] for c in mod.columns]
      
    return mod

In [ ]:
data = parseProteomicsDataset(projectID = 1, uri = "../Data/NAFLD/proteinGroups.txt")

In [ ]:
data

In [ ]:
extractModificationProteinSubjectRelationships(data)

In [ ]:
aux